# Laboratory 4 — Hodgkin–Huxley Mechanisms

Companion to Chapter 3. Uses mV, ms, current density µA/cm², conductance density mS/cm², and capacitance density µF/cm². Classical squid-axon kinetics are used without temperature scaling (6.3 °C). No threshold/reset generates the voltage waveform.

Run all cells, then change one parameter at a time. Figures are synthetic teaching outputs.

[Run in Colab](https://colab.research.google.com/github/an-hongyu/Introduction-to-Neuromorphic-Embodied-AI/blob/main/notebooks/hodgkin_huxley.ipynb)

## 1. Model and numerical convention

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def hh_rates(v):
    # Absolute V in mV; rates in ms^-1. Removable singularities use limits.
    def ratio(x, scale):
        return scale * (1 + x / (2 * scale)) if abs(x / scale) < 1e-7 else x / (-np.expm1(-x / scale))
    return np.array([0.1 * ratio(v + 40, 10), 4 * np.exp(-(v + 65) / 18),
                     0.07 * np.exp(-(v + 65) / 20), 1 / (1 + np.exp(-(v + 35) / 10)),
                     0.01 * ratio(v + 55, 10), 0.125 * np.exp(-(v + 65) / 80)])

def simulate_hh(drive=10.0, dt=0.01, duration=65.0, g_na=120.0, g_k=36.0):
    """Single compartment; mV, ms, uA/cm^2, mS/cm^2, uF/cm^2.

    Classical squid-axon kinetics at 6.3 deg C; no temperature rescaling.
    Step current from 10 to 50 ms. RK4, held current per interval; no reset.
    """
    t = np.arange(round(duration / dt) + 1) * dt
    current = np.where((t >= 10) & (t < 50), drive, 0.0)
    rates = hh_rates(-65)
    state = np.empty((len(t), 4))
    state[0] = [-65, rates[0] / sum(rates[:2]), rates[2] / sum(rates[2:4]), rates[4] / sum(rates[4:])]
    def derivative(s, injected):
        v, m, h, n = s
        am, bm, ah, bh, an, bn = hh_rates(v)
        ina = g_na * m**3 * h * (v - 50)
        ik = g_k * n**4 * (v + 77)
        il = 0.3 * (v + 54.387)
        return np.array([injected - ina - ik - il,
                         am * (1 - m) - bm * m, ah * (1 - h) - bh * h,
                         an * (1 - n) - bn * n])
    for k in range(len(t) - 1):
        s, i = state[k], current[k]
        a = derivative(s, i); b = derivative(s + dt * a / 2, i)
        c = derivative(s + dt * b / 2, i); d = derivative(s + dt * c, i)
        state[k + 1] = s + dt * (a + 2*b + 2*c + d) / 6
    v, m, h, n = state.T
    currents = np.column_stack([g_na*m**3*h*(v-50), g_k*n**4*(v+77), .3*(v+54.387)])
    up = np.flatnonzero((v[:-1] < 0) & (v[1:] >= 0))
    events = t[up] + dt * (-v[up]) / (v[up + 1] - v[up])
    return t, current, state, currents, events

## 2. Change the applied current and channel conductances

In [ ]:
# EDIT drive, g_na, or g_k, then rerun this cell.
t, current, state, ionic, events = simulate_hh(drive=10, g_na=120, g_k=36)
fig, ax = plt.subplots(4, 1, figsize=(10, 9), sharex=True, constrained_layout=True)
ax[0].plot(t, current, color='#24628a'); ax[0].set(ylabel='Input (µA/cm²)', title='Hodgkin–Huxley: current, voltage, gates, and ionic currents')
ax[1].plot(t, state[:, 0], color='#24628a'); ax[1].set(ylabel='Voltage (mV)')
for k, name in enumerate(['m: Na activation', 'h: Na inactivation', 'n: K activation']): ax[2].plot(t, state[:, k+1], label=name)
ax[2].set(ylabel='Gate state', ylim=(0, 1)); ax[2].legend(fontsize=9, loc='upper right')
for k, name in enumerate(['Na', 'K', 'leak']): ax[3].plot(t, ionic[:, k], label=name)
ax[3].set(xlabel='Time (ms)', ylabel='Outward current (µA/cm²)'); ax[3].legend(fontsize=9)
plt.show()
print('Upward 0 mV crossings (ms):', np.round(events, 3))

## 3. Sodium ablation and numerical checks

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True, constrained_layout=True,
                       gridspec_kw={'height_ratios': [3, 1]})
for g_na, color, label in [(120, '#24628a', 'Standard conductance'), (0, '#b75d45', 'Na conductance set to zero')]:
    t, _, s, _, e = simulate_hh(g_na=g_na)
    ax[0].plot(t, s[:, 0], color=color, label=label)
    ax[1].vlines(e, 0.1 if g_na else 0.6, 0.4 if g_na else 0.9, color=color)
ax[0].set(ylabel='Voltage (mV)', title='A parameter ablation, not a simulated drug dose'); ax[0].legend(fontsize=9)
ax[1].set(xlabel='Time (ms)', ylabel='Events', yticks=[], ylim=(0, 1))
plt.show()

_, _, quiet, _, no_events = simulate_hh(drive=0)
assert len(no_events) == 0 and np.max(np.abs(quiet[:, 0] + 65)) < 0.1
_, _, normal, _, e = simulate_hh()
_, _, blocked, _, eb = simulate_hh(g_na=0)
assert len(e) >= 2 and len(eb) == 0 and normal[:, 0].max() > 30
assert np.all((normal[:, 1:] >= 0) & (normal[:, 1:] <= 1))
assert np.isfinite(hh_rates(-40)).all() and np.isfinite(hh_rates(-55)).all()
ef = simulate_hh(dt=.005)[4]
assert len(ef) == len(e) and np.max(np.abs(e - ef)) < .05
print('HH checks passed: rest, gate bounds, Na ablation, singularities, and time-step refinement.')

## References

[1] A. L. Hodgkin and A. F. Huxley, “A quantitative description of membrane current and its application to conduction and excitation in nerve,” *J. Physiol.*, vol. 117, pp. 500–544, 1952. [DOI](https://doi.org/10.1113/jphysiol.1952.sp004764).

[2] Brian 2 developers, “Hodgkin–Huxley equations (1952).” [Reference implementation](https://brian2.readthedocs.io/en/stable/examples/compartmental.hodgkin_huxley_1952.html). This notebook shifts the voltage origin by −65 mV, uses one compartment, initializes gates at equilibrium, and uses RK4; it is not a reproduction of the Brian cable experiment.